# Discretization Verification

Numerical verification of the model's state and quadrature discretization.

**Two domains, two questions per domain:**
- **A. Returns / financial state** â€” 3-D financial state grid (`y_1, spr, cy`), state innovation quadrature on `Sigma_ss`, return residual quadrature on `Sigma_r_cond`.
- **B. Labour / income** â€” `z`-grid, persistent innovation `eta` (mixture-normal), transitory `eps` (mixture-normal).
- **Correctness:** does the discretization reproduce the moments and identities the model claims?
- **Convergence:** what does the user gain (accuracy) and lose (cost) at each dial?

This notebook **diagnoses** the frozen discretization in `discretization.py` / `precompute.py`. It does not modify them.


## Â§0  Setup

Imports, build a production-spec `Precompute`, define helpers used throughout.


In [ ]:
# Imports
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.linalg as la
from scipy.special import roots_hermite
from scipy.signal import lfilter
from scipy.stats import norm
import matplotlib.pyplot as plt

from var import build_nominal_system1_var_config
from precompute import build_model, Precompute
from model import DiscretizationConfig, SolverConfig
from discretization import (
    build_state_grid, stationary_covariance,
    get_state_quadrature, get_return_quadrature,
    get_eta_quadrature_mixture, get_eps_quadrature_corrected,
    discretize_income_ar1_mixture,
)
from solver import run_lifecycle_solver, transform_state_for_bracketing_3d
from simulation import simulate_lifecycle

np.set_printoptions(suppress=True, linewidth=140, precision=6)
warnings.filterwarnings("ignore", message="The behavior of DatetimeProperties.to_pydatetime")

# Resolve data path robustly
_path_candidates = [Path("data/var_dataset.csv"), Path("../data/var_dataset.csv")]
_VAR_CSV = next((p for p in _path_candidates if p.exists()), None)
if _VAR_CSV is None:
    raise FileNotFoundError("Could not find data/var_dataset.csv")

print(f"Using VAR data: {_VAR_CSV}")


In [ ]:
# Production base configuration (Catherine 2025 calibration)
BASE_CFG = {
    "beta": 0.96, "gamma": 3.0, "b_bar": 10,
    "start_age": 22, "retire_age": 67, "terminal_age": 99,
    "b0": -6.142, "b1": 0.3040, "b2": -0.051, "b3": 0.002586,
    "rho": 0.991, "pz": 0.176,
    "mu_eta1": -0.524, "sigma_eta1": 0.113,
    "mu_eta2": -(0.176 / (1.0 - 0.176)) * (-0.524),  # informational; quadrature recomputes
    "sigma_eta2": 0.046,
    "pe": 0.044, "mu_eps1": 0.134, "sigma_eps1": 0.762,
    "mu_eps2": 0.0, "sigma_eps2": 0.055,
    "constrained": True,
}

# Production discretization.
# NOTE: under the Judd-mixture quadrature, n_eta_nodes / n_eps_nodes are the
# TOTAL node count (no longer per-component K).  Polynomial exactness is
# 2*n - 1 against the mixture density.  Defaults below give exactness 5
# (eta) and 9 (eps); see Â§B.2/Â§B.3 below for the derivation and Â§C.4/Â§C.5
# for the CRRA-integrand stress test.
PROD_DISC = DiscretizationConfig(
    n_wealth=150, n_savings=150,
    state_grid_sizes=(7, 7, 7),
    state_grid_mode="principal",
    state_n_stds=3.0,
    n_z=11, n_stds=3.0,
    n_eta_nodes=3, n_eps_nodes=5,
    n_ret_nodes_1d=3, n_state_quad_nodes=3,
)

VAR_CONFIG, _, VAR_DATA = build_nominal_system1_var_config(csv_path=str(_VAR_CSV))
MODEL = build_model(BASE_CFG, VAR_CONFIG, verbose=False)
PC = Precompute(MODEL, PROD_DISC, verbose=False)

print(f"Production Precompute built.")
print(f"  state_grid: {PC.state_grid.shape}  ({PC.N_state} joint states, mode='{PC.state_grid_mode}')")
print(f"  z_grid:     {PC.z_grid.shape}      n_eta={len(PC.eta_nodes)}  n_eps={len(PC.eps_nodes)}")
print(f"  v_nodes:    {PC.v_nodes.shape}     ret_nodes: {PC.ret_nodes.shape}")
print(f"  Historical sample rows: {len(VAR_DATA)}")


In [ ]:
# Helpers
def _verdict(passed, name, residual=None, tol=None):
    tag = "PASS" if passed else "FAIL"
    extra = ""
    if residual is not None:
        extra = f"  residual = {residual:.2e}" + (f"  (tol = {tol:.0e})" if tol is not None else "")
    print(f"  [{tag}] {name}{extra}")
    return bool(passed)


def check(name, condition, residual=None, tol=None):
    return _verdict(bool(condition), name, residual=residual, tol=tol)


def section_summary(results, title):
    n = len(results)
    n_pass = sum(results)
    print()
    if n_pass == n:
        print(f"  ===> SECTION VERDICT: {title}: ALL {n}/{n} TIER-1 CHECKS PASS")
    else:
        print(f"  ===> SECTION VERDICT: {title}: {n_pass}/{n} pass; {n - n_pass} FAILED")
    return n_pass, n


def mixture_moments(p1, m1, s1, p2, m2, s2):
    """Closed-form moments of a two-component normal mixture."""
    p2_w = 1.0 - p1
    mu = p1 * m1 + p2_w * m2
    var = p1 * (s1 ** 2 + (m1 - mu) ** 2) + p2_w * (s2 ** 2 + (m2 - mu) ** 2)
    m3 = (p1 * ((m1 - mu) ** 3 + 3.0 * (m1 - mu) * s1 ** 2)
          + p2_w * ((m2 - mu) ** 3 + 3.0 * (m2 - mu) * s2 ** 2))
    m4 = (p1 * ((m1 - mu) ** 4 + 6.0 * (m1 - mu) ** 2 * s1 ** 2 + 3.0 * s1 ** 4)
          + p2_w * ((m2 - mu) ** 4 + 6.0 * (m2 - mu) ** 2 * s2 ** 2 + 3.0 * s2 ** 4))
    skew = m3 / var ** 1.5
    excess_kurt = m4 / var ** 2 - 3.0
    return mu, var, skew, excess_kurt


def quad_moments(nodes, weights):
    """Empirical (weighted) moments from a univariate quadrature rule."""
    w = np.asarray(weights)
    x = np.asarray(nodes)
    m = float(np.sum(w * x))
    var = float(np.sum(w * (x - m) ** 2))
    if var <= 0:
        return m, var, np.nan, np.nan
    skew = float(np.sum(w * (x - m) ** 3) / var ** 1.5)
    excess_kurt = float(np.sum(w * (x - m) ** 4) / var ** 2 - 3.0)
    return m, var, skew, excess_kurt


def trilinear_u(u_pt, u_grids, values_flat, N_vec):
    """Trilinear interpolation in the bracket-grid (u) coordinate system."""
    lo, frac = [], []
    for d in range(3):
        g = u_grids[d]
        n = len(g)
        idx = int(np.searchsorted(g, u_pt[d]) - 1)
        idx = max(0, min(idx, n - 2))
        f = (u_pt[d] - g[idx]) / (g[idx + 1] - g[idx])
        f = max(0.0, min(1.0, f))
        lo.append(idx)
        frac.append(f)
    out = 0.0
    N0, N1, N2 = N_vec
    for d0 in range(2):
        for d1 in range(2):
            for d2 in range(2):
                w = ((1.0 - frac[0]) if d0 == 0 else frac[0]) * \
                    ((1.0 - frac[1]) if d1 == 0 else frac[1]) * \
                    ((1.0 - frac[2]) if d2 == 0 else frac[2])
                j = (lo[0] + d0) * N1 * N2 + (lo[1] + d1) * N2 + (lo[2] + d2)
                out += w * values_flat[j]
    return out


# ----- Performance budget switches -----
# Tier-1 / Tier-2 / integrand-only sweeps always run (cheap).
# Full-solver sweeps below are gated.  Switch to True for the full study.
RUN_FULL_SOLVER_SWEEPS = False    # gate for sections C.1, C.6, C.7

print("Helpers defined.  RUN_FULL_SOLVER_SWEEPS =", RUN_FULL_SOLVER_SWEEPS)


## Â§A  Returns / financial-state verification

Verifies the 3-D financial-state grid and the two return-side quadrature rules.


### Â§A.1  VAR and Lyapunov consistency  (Tier 1, exact)

Sanity checks on the VAR partition: stationarity, mean recovery, Lyapunov fixed point,
PSD of all relevant covariances, and the conditional-covariance identity
`Sigma_rr = Sigma_r_cond + M Sigma_ss M^T`.


In [ ]:
# A.1  VAR / Lyapunov consistency
results = []
mu_s = MODEL.z_bar_state
Phi_11 = np.asarray(MODEL.Phi_11)
Phi_0_state = np.asarray(MODEL.Phi_0_state)
Sigma_ss = np.asarray(MODEL.Sigma_ss)
Sigma_rr = np.asarray(MODEL.Sigma_rr)
Sigma_rs = np.asarray(MODEL.Sigma_rs)
Sigma_sr = Sigma_rs.T
M = np.asarray(MODEL.M)
Sigma_r_cond = np.asarray(MODEL.Sigma_r_cond)
Sigma_z = stationary_covariance(Phi_11, Sigma_ss)

# (I - Phi_11)^{-1} Phi_0 = mu_s
mu_recovered = np.linalg.solve(np.eye(3) - Phi_11, Phi_0_state)
err_mean = float(np.max(np.abs(mu_recovered - mu_s)))
results.append(check("(I - Phi_11)^-1 Phi_0_state == z_bar_state", err_mean < 1e-13, err_mean, 1e-13))

# Lyapunov: Sigma_z = Phi_11 Sigma_z Phi_11^T + Sigma_ss
lyap_residual = Sigma_z - Phi_11 @ Sigma_z @ Phi_11.T - Sigma_ss
err_lyap = float(np.max(np.abs(lyap_residual)))
results.append(check("Lyapunov residual", err_lyap < 1e-12, err_lyap, 1e-12))

# Sigma_z symmetric and PSD
sym_sz = float(np.max(np.abs(Sigma_z - Sigma_z.T)))
eigs_sz = np.linalg.eigvalsh(0.5 * (Sigma_z + Sigma_z.T))
results.append(check("Sigma_z symmetric", sym_sz < 1e-14, sym_sz, 1e-14))
results.append(check("Sigma_z PSD (min eig >= 0)", float(eigs_sz.min()) >= -1e-14, float(eigs_sz.min())))

# Sigma_r_cond = Sigma_rr - M Sigma_sr  symmetric and PSD
S_cond_check = Sigma_rr - M @ Sigma_sr
err_cond_def = float(np.max(np.abs(S_cond_check - Sigma_r_cond)))
results.append(check("Sigma_r_cond definition", err_cond_def < 1e-15, err_cond_def, 1e-15))
sym_rc = float(np.max(np.abs(Sigma_r_cond - Sigma_r_cond.T)))
eigs_rc = np.linalg.eigvalsh(0.5 * (Sigma_r_cond + Sigma_r_cond.T))
results.append(check("Sigma_r_cond symmetric", sym_rc < 1e-14, sym_rc, 1e-14))
results.append(check("Sigma_r_cond PSD (min eig >= 0)", float(eigs_rc.min()) >= -1e-14, float(eigs_rc.min())))

# Sigma_rr = Sigma_r_cond + M Sigma_ss M^T
err_decomp = float(np.max(np.abs(Sigma_rr - (Sigma_r_cond + M @ Sigma_ss @ M.T))))
results.append(check("Sigma_rr = Sigma_r_cond + M Sigma_ss M^T", err_decomp < 1e-15, err_decomp, 1e-15))

# Phi_11 stationary
max_eig = float(np.max(np.abs(np.linalg.eigvals(Phi_11))))
results.append(check("max |eig(Phi_11)| < 1", max_eig < 1.0 - 1e-6, 1.0 - max_eig))

section_summary(results, "A.1 VAR / Lyapunov consistency")


### Â§A.2  State grid geometry  (Tier 1, exact)

For each grid mode at `N=(7,7,7), n_stds=3.0`: centroid identity, coordinate round-trip,
flat indexing, and bracket transform consistency on every grid point.


In [ ]:
# A.2  State grid geometry across modes
results = []
N_VEC = (7, 7, 7)
N_TOT = int(np.prod(N_VEC))
for mode in ("naive", "lyapunov-axis", "principal"):
    g = build_state_grid(N_vec=N_VEC, mu_intercept=Phi_0_state,
                         Phi=Phi_11, Sigma_innov=Sigma_ss,
                         n_stds=3.0, mode=mode)
    s_grid = g["state_grid"]
    indices = g["state_indices"]
    L = g["L"]; L_inv = g["L_inv"]; m = g["mu_s"]
    bracket_shift = g["bracket_shift"]
    bracket_L_inv = g["bracket_L_inv"]
    bracket_grids = g["state_bracket_grids"]

    # Centroid sanity
    centre_flat = (3, 3, 3)
    centre_idx = centre_flat[0] * N_VEC[1] * N_VEC[2] + centre_flat[1] * N_VEC[2] + centre_flat[2]
    if mode == "principal":
        results.append(check(f"[{mode}] centroid grid point == mu_s",
                             np.allclose(s_grid[centre_idx], m, atol=1e-12),
                             float(np.max(np.abs(s_grid[centre_idx] - m))), 1e-12))
        # state_grid[i] = mu_s + L @ u_lattice[i]  for every i
        u_lattice = np.empty((N_TOT, 3))
        for i in range(N_TOT):
            u_lattice[i] = np.array([bracket_grids[d][indices[i, d]] for d in range(3)])
        reconstructed = m[None, :] + u_lattice @ L.T
        err_recon = float(np.max(np.abs(reconstructed - s_grid)))
        results.append(check(f"[{mode}] state_grid == mu_s + L @ u_lattice",
                             err_recon < 1e-13, err_recon, 1e-13))
        # Round-trip: u = L^{-1} (s - mu_s)
        u_recovered = (s_grid - m) @ L_inv.T
        err_round = float(np.max(np.abs(u_recovered - u_lattice)))
        results.append(check(f"[{mode}] L_inv (s - mu_s) round-trip",
                             err_round < 1e-12, err_round, 1e-12))
    else:
        # Naive / lyapunov-axis: centred axes (s_grid centroid equals axis midpoints)
        midpoint = np.array([0.5 * (bg[0] + bg[-1]) for bg in bracket_grids])
        err_centre = float(np.max(np.abs(s_grid[centre_idx] - midpoint)))
        results.append(check(f"[{mode}] grid centroid is axis midpoint",
                             err_centre < 1e-12, err_centre, 1e-12))
        # bracket_shift = 0 and bracket_L_inv = I
        results.append(check(f"[{mode}] bracket_shift == 0",
                             float(np.max(np.abs(bracket_shift))) < 1e-15,
                             float(np.max(np.abs(bracket_shift)))))
        results.append(check(f"[{mode}] bracket_L_inv == I",
                             np.allclose(bracket_L_inv, np.eye(3), atol=1e-15),
                             float(np.max(np.abs(bracket_L_inv - np.eye(3))))))

    # Flat indexing: i = i0 * N1 * N2 + i1 * N2 + i2
    flat_ok = True
    for i in range(N_TOT):
        i0, i1, i2 = indices[i]
        if i != i0 * N_VEC[1] * N_VEC[2] + i1 * N_VEC[2] + i2:
            flat_ok = False
            break
    results.append(check(f"[{mode}] flat indexing", flat_ok))

    # Bracket transform on each grid point reproduces a {0,1} fraction
    grids_d = [np.asarray(bracket_grids[d]) for d in range(3)]
    max_frac_residual = 0.0
    for i in range(N_TOT):
        s_i = s_grid[i]
        u0, u1, u2 = transform_state_for_bracketing_3d(
            float(s_i[0]), float(s_i[1]), float(s_i[2]),
            np.asarray(bracket_shift), np.asarray(bracket_L_inv),
        )
        u_pt = np.array([u0, u1, u2])
        for d in range(3):
            g_d = grids_d[d]
            # find the lattice axis position
            pos = (u_pt[d] - g_d[0]) / (g_d[1] - g_d[0])
            err = abs(pos - round(pos))
            if err > max_frac_residual:
                max_frac_residual = err
    results.append(check(f"[{mode}] bracket transform on grid points -> integer u-coord",
                         max_frac_residual < 1e-10, max_frac_residual, 1e-10))

section_summary(results, "A.2 state grid geometry")


### Â§A.3  State innovation quadrature  (Tier 1)

Sweep `K_s in {2, 3, 4, 5}`. Verify `sum w = 1`, `sum w v = 0`,
`sum w v v^T = Sigma_ss`, and positive weights.


In [ ]:
# A.3  State innovation quadrature moments  (sweep K_s)
results = []
rows = []
for K_s in (2, 3, 4, 5):
    v_n, v_w = get_state_quadrature(MODEL, n_nodes=K_s)
    sw = float(v_w.sum())
    mean_v = v_w @ v_n              # (3,)
    Sigma_emp = (v_n.T * v_w) @ v_n  # weighted outer product
    err_sum = abs(sw - 1.0)
    err_mean = float(np.max(np.abs(mean_v)))
    err_cov = float(np.max(np.abs(Sigma_emp - Sigma_ss)))
    pos_w = bool(np.all(v_w > 0))
    K_tot = K_s ** 3
    rows.append((K_s, K_tot, err_sum, err_mean, err_cov, pos_w))
    results.append(check(f"K_s={K_s}  sum w == 1", err_sum < 1e-15, err_sum, 1e-15))
    results.append(check(f"K_s={K_s}  E[v] == 0", err_mean < 1e-15, err_mean, 1e-15))
    results.append(check(f"K_s={K_s}  E[v v^T] == Sigma_ss", err_cov < 1e-14, err_cov, 1e-14))
    results.append(check(f"K_s={K_s}  all weights > 0", pos_w))

df = pd.DataFrame(rows, columns=["K_s", "K_total", "|sum w - 1|", "|E[v]|_inf", "|cov err|_inf", "all w>0"])
print(); print(df.to_string(index=False))
section_summary(results, "A.3 state innovation quadrature")


### Â§A.4  Return residual quadrature  (Tier 1)

Sweep `K_r in {2, 3, 4, 5}` against `Sigma_r_cond`.


In [ ]:
# A.4  Return residual quadrature moments  (sweep K_r)
results = []
rows = []
for K_r in (2, 3, 4, 5):
    r_n, r_w = get_return_quadrature(MODEL, n_nodes=K_r)
    sw = float(r_w.sum())
    mean_r = r_w @ r_n
    Sigma_emp = (r_n.T * r_w) @ r_n
    err_sum = abs(sw - 1.0)
    err_mean = float(np.max(np.abs(mean_r)))
    err_cov = float(np.max(np.abs(Sigma_emp - Sigma_r_cond)))
    pos_w = bool(np.all(r_w > 0))
    rows.append((K_r, K_r ** 3, err_sum, err_mean, err_cov, pos_w))
    results.append(check(f"K_r={K_r}  sum w == 1", err_sum < 1e-15, err_sum, 1e-15))
    results.append(check(f"K_r={K_r}  E[r] == 0", err_mean < 1e-15, err_mean, 1e-15))
    results.append(check(f"K_r={K_r}  E[r r^T] == Sigma_r_cond", err_cov < 1e-14, err_cov, 1e-14))
    results.append(check(f"K_r={K_r}  all weights > 0", pos_w))

df = pd.DataFrame(rows, columns=["K_r", "K_total", "|sum w - 1|", "|E[r]|_inf", "|cov err|_inf", "all w>0"])
print(); print(df.to_string(index=False))
section_summary(results, "A.4 return residual quadrature")


### Â§A.5  Joint quadrature consistency  (Tier 1)

Joint state x return integration must reproduce the *full* unconditional return
covariance `Sigma_rr` and cross-covariance `Sigma_rs`.


In [ ]:
# A.5  Joint state x return quadrature: covariance reproduction
results = []
v_n, v_w = get_state_quadrature(MODEL, n_nodes=PROD_DISC.n_state_quad_nodes)
r_n, r_w = get_return_quadrature(MODEL, n_nodes=PROD_DISC.n_ret_nodes_1d)
M_v = v_n @ M.T   # (K_s_total, n_ret)

# Unconditional return covariance: x_kj = M v_k + r_j;  E[x x^T] = M Sigma_ss M^T + Sigma_r_cond = Sigma_rr
S_emp = np.zeros((3, 3))
S_cross_emp = np.zeros((3, 3))
for k_v in range(len(v_w)):
    for k_r in range(len(r_w)):
        x = M_v[k_v] + r_n[k_r]
        ww = v_w[k_v] * r_w[k_r]
        S_emp += ww * np.outer(x, x)
        S_cross_emp += ww * np.outer(x, v_n[k_v])
err_rr = float(np.max(np.abs(S_emp - Sigma_rr)))
err_rs = float(np.max(np.abs(S_cross_emp - Sigma_rs)))
results.append(check("E[(Mv + r)(Mv + r)^T] == Sigma_rr", err_rr < 1e-14, err_rr, 1e-14))
results.append(check("E[(Mv + r) v^T] == Sigma_rs",       err_rs < 1e-14, err_rs, 1e-14))

# Conditional return mean at every grid point
max_err = 0.0
for i in range(PC.N_state):
    s_i = PC.state_grid[i]
    base_mu_r = MODEL.Phi_0_ret + MODEL.Phi_21 @ s_i
    avg_mu_r = base_mu_r + (v_w @ M_v)
    target = MODEL.Phi_0_ret + MODEL.Phi_21 @ s_i
    err = float(np.max(np.abs(avg_mu_r - target)))
    if err > max_err:
        max_err = err
results.append(check("E_v[base_mu_r_i + M v] == Phi_0_ret + Phi_21 s_i  (all i)",
                     max_err < 1e-12, max_err, 1e-12))

section_summary(results, "A.5 joint quadrature consistency")


### Â§A.6  Recovery identity  (Tier 1, in-solver formula)

Reproduce the solver's three-asset return formula at one grid point and one
quadrature node, and confirm the residual node maps cleanly onto each asset.


In [ ]:
# A.6  Recovery identity at one (i, k_v, k_r) tuple
results = []
i = PC.N_state // 2
k_v = len(PC.v_weights) // 2
k_r = len(PC.ret_weights) // 2

s_i = PC.state_grid[i]
base_mu_r = PC.const_r + PC.A_r @ s_i
mu_r_node = base_mu_r + PC.M_v_nodes[k_v]   # (3,)
res = PC.ret_nodes[k_r]
R_bill  = float(np.exp(mu_r_node[0] + res[0]))
R_stock = R_bill * float(np.exp(mu_r_node[1] + res[1]))
R_bond  = R_bill * float(np.exp(mu_r_node[2] + res[2]))

err_xr = abs((np.log(R_stock) - np.log(R_bill)) - mu_r_node[1] - res[1])
err_xb = abs((np.log(R_bond)  - np.log(R_bill)) - mu_r_node[2] - res[2])
err_rtb = abs(np.log(R_bill) - mu_r_node[0] - res[0])
results.append(check("log R_bill - mu_r[0] - r[0] == 0",  err_rtb < 1e-15, err_rtb, 1e-15))
results.append(check("log(R_stock/R_bill) - mu_r[1] - r[1] == 0", err_xr < 1e-15, err_xr, 1e-15))
results.append(check("log(R_bond /R_bill) - mu_r[2] - r[2] == 0", err_xb < 1e-15, err_xb, 1e-15))

print(f"  s_i = {s_i}")
print(f"  R_bill={R_bill:.6f}  R_stock={R_stock:.6f}  R_bond={R_bond:.6f}")
section_summary(results, "A.6 recovery identity")


### Â§A.7  Coverage  (Tier 2)

Coverage of the historical sample and of a 200,000-draw stationary Monte
Carlo, across `(mode, n_stds)`.  Hull volume reported as ratio vs the
naive baseline at the same `n_stds`.


In [ ]:
# A.7  Coverage
hist = VAR_DATA[["y_1", "spr", "cy"]].to_numpy()
rng = np.random.default_rng(20260427)
mc_z = rng.multivariate_normal(mean=mu_s, cov=Sigma_z, size=200_000)

modes = ["naive", "lyapunov-axis", "principal"]
n_stds_grid = [2.0, 2.5, 3.0, 3.5]

coverage_rows = []
for mode in modes:
    # naive baseline volume at each n_stds (denominator for ratio)
    baseline_vol = {}
    for nstd in n_stds_grid:
        g_naive = build_state_grid(N_vec=N_VEC, mu_intercept=Phi_0_state,
                                   Phi=Phi_11, Sigma_innov=Sigma_ss,
                                   n_stds=nstd, mode="naive")
        bg = g_naive["state_bracket_grids"]
        baseline_vol[nstd] = float(np.prod([bg[d][-1] - bg[d][0] for d in range(3)]))
    for nstd in n_stds_grid:
        g = build_state_grid(N_vec=N_VEC, mu_intercept=Phi_0_state,
                             Phi=Phi_11, Sigma_innov=Sigma_ss,
                             n_stds=nstd, mode=mode)
        bg = g["state_bracket_grids"]
        if mode == "principal":
            # bracket-cube test: |L^{-1}(s - mu_s)|_inf <= nstd
            L_inv = g["L_inv"]; m_ = g["mu_s"]
            inside_hist = float(np.mean(np.all(np.abs((hist - m_) @ L_inv.T) <= nstd, axis=1)))
            inside_mc = float(np.mean(np.all(np.abs((mc_z - m_) @ L_inv.T) <= nstd, axis=1)))
            vol = (2.0 * nstd) ** 3 * abs(np.linalg.det(g["L"]))
            max_u = float(np.max(np.abs((hist - m_) @ L_inv.T)))
        else:
            lo = np.array([bg[d][0] for d in range(3)])
            hi = np.array([bg[d][-1] for d in range(3)])
            inside_hist = float(np.mean(np.all((hist >= lo) & (hist <= hi), axis=1)))
            inside_mc = float(np.mean(np.all((mc_z >= lo) & (mc_z <= hi), axis=1)))
            vol = float(np.prod(hi - lo))
            sigma_z_diag = g["sigma_z"]
            max_u = float(np.max(np.abs(hist - g["mu_s"]) / sigma_z_diag))
        coverage_rows.append({
            "mode": mode, "n_stds": nstd,
            "hist_in_pct": 100.0 * inside_hist,
            "mc_in_pct": 100.0 * inside_mc,
            "hull_vol": vol,
            "vol_ratio_vs_naive": vol / baseline_vol[nstd],
            "max_u_hist": max_u,
        })

cov_df = pd.DataFrame(coverage_rows)
print(cov_df.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))

# Plot: stationary MC scatter (y_1 vs cy) with principal grid hull at n_stds=3.0
g3 = build_state_grid(N_vec=N_VEC, mu_intercept=Phi_0_state, Phi=Phi_11,
                      Sigma_innov=Sigma_ss, n_stds=3.0, mode="principal")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (xidx, yidx, xlbl, ylbl) in zip(axes, [(0, 2, "y_1", "cy"), (0, 1, "y_1", "spr")]):
    ax.scatter(mc_z[:5000, xidx], mc_z[:5000, yidx], s=2, alpha=0.2, label="stationary MC")
    ax.scatter(hist[:, xidx], hist[:, yidx], s=15, color="C3", label="historical")
    ax.scatter(g3["state_grid"][:, xidx], g3["state_grid"][:, yidx],
               s=20, marker="x", color="k", label="principal grid")
    ax.set_xlabel(xlbl); ax.set_ylabel(ylbl); ax.legend(loc="best", fontsize=8)
fig.suptitle("Coverage: stationary MC, historical, and principal-mode grid (N=7^3, n_stds=3)")
plt.tight_layout(); plt.show()

print("Coverage check: principal-mode at n_stds=3.0 covers historical sample at",
      f"{cov_df[(cov_df['mode']=='principal') & (cov_df['n_stds']==3.0)]['hist_in_pct'].iloc[0]:.1f}%")


### Â§A.8  Trilinear interpolation accuracy  (Tier 2)

Linear functions: trilinear interpolation in u-space is **exact** at machine precision.
Smooth nonlinear functions: error decays as `O(1/N^2)`.


In [ ]:
# A.8  Trilinear interpolation: exactness on linear, convergence on smooth
results = []

# (a) Linear function -- exact at any N
g_p = build_state_grid(N_vec=(7, 7, 7), mu_intercept=Phi_0_state,
                       Phi=Phi_11, Sigma_innov=Sigma_ss, n_stds=3.0, mode="principal")
mu_local = g_p["mu_s"]; L_local = g_p["L"]; L_inv_local = g_p["L_inv"]
u_grids = g_p["state_bracket_grids"]
N_vec_local = (7, 7, 7)

a, b = 1.234, np.array([0.3, -0.5, 0.1])
def f_lin(s):
    return a + b @ s

values = np.array([f_lin(s) for s in g_p["state_grid"]])
rng = np.random.default_rng(7)
max_err_lin = 0.0
for _ in range(200):
    u_pt = rng.uniform(-3.0, 3.0, size=3)
    s_pt = mu_local + L_local @ u_pt
    interp = trilinear_u(u_pt, u_grids, values, N_vec_local)
    exact = f_lin(s_pt)
    max_err_lin = max(max_err_lin, abs(interp - exact))
results.append(check("Trilinear exact on linear function (N=7)",
                     max_err_lin < 1e-12, max_err_lin, 1e-12))

# (b) Smooth nonlinear function -- convergence
def f_smooth(s):
    return float(np.exp(s[0]) + s[1] * s[2])

N_sweep = (5, 7, 9, 11)
err_rows = []
for N in N_sweep:
    g_n = build_state_grid(N_vec=(N, N, N), mu_intercept=Phi_0_state,
                           Phi=Phi_11, Sigma_innov=Sigma_ss, n_stds=3.0, mode="principal")
    vals_n = np.array([f_smooth(s) for s in g_n["state_grid"]])
    rng2 = np.random.default_rng(7)
    Linf = 0.0; L2sum = 0.0
    n_pts = 200
    for _ in range(n_pts):
        u_pt = rng2.uniform(-3.0, 3.0, size=3)
        s_pt = g_n["mu_s"] + g_n["L"] @ u_pt
        interp = trilinear_u(u_pt, g_n["state_bracket_grids"], vals_n, (N, N, N))
        exact = f_smooth(s_pt)
        e = abs(interp - exact)
        Linf = max(Linf, e); L2sum += e ** 2
    err_rows.append({"N": N, "L_inf": Linf, "L2_rms": np.sqrt(L2sum / n_pts)})

err_df = pd.DataFrame(err_rows)
print(err_df.to_string(index=False, float_format=lambda v: f"{v:.4e}"))

# Convergence plot
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(err_df["N"], err_df["L_inf"], "o-", label="L_inf err")
ax.loglog(err_df["N"], err_df["L2_rms"], "s-", label="L2 RMS err")
N_arr = np.array(N_sweep, float)
ax.loglog(N_arr, err_df["L_inf"].iloc[0] * (N_arr[0] / N_arr) ** 2, "--",
          color="grey", alpha=0.6, label="O(1/N^2) reference")
ax.set_xlabel("N (per-axis grid size)"); ax.set_ylabel("interp error")
ax.legend(); ax.set_title("Trilinear interpolation error on f(s)=exp(s0) + s1 s2")
plt.tight_layout(); plt.show()

# Convergence rate from N=7 -> N=11
ratio = err_df["L_inf"].iloc[1] / err_df["L_inf"].iloc[3]
results.append(check("Smooth-function L_inf err shrinks N=7 -> N=11",
                     ratio > 1.5, ratio))

section_summary(results, "A.8 trilinear interpolation accuracy")


## Â§B  Labour / income verification

Verifies the persistent and transitory income innovation quadratures, the
mean-zero enforcement, the z-grid stationarity, and the income lookup table.


### Â§B.1  Mixture parameter consistency  (Tier 1)

Closed-form moments of the calibrated `eta` and `eps` mixtures, with the
mean-zero enforcement on component 2.


In [ ]:
# B.1  Mixture moments (closed-form)
results = []
pz, mu_eta1, sigma_eta1, sigma_eta2 = MODEL.pz, MODEL.mu_eta1, MODEL.sigma_eta1, MODEL.sigma_eta2
pe, mu_eps1, sigma_eps1, sigma_eps2 = MODEL.pe, MODEL.mu_eps1, MODEL.sigma_eps1, MODEL.sigma_eps2

mu_eta2_eff = -(pz / (1.0 - pz)) * mu_eta1
mu_eps2_eff = -(pe / (1.0 - pe)) * mu_eps1

mean_eta_check = pz * mu_eta1 + (1.0 - pz) * mu_eta2_eff
mean_eps_check = pe * mu_eps1 + (1.0 - pe) * mu_eps2_eff
results.append(check("Effective mu_eta2 enforces E[eta]=0",
                     abs(mean_eta_check) < 1e-15, abs(mean_eta_check), 1e-15))
results.append(check("Effective mu_eps2 enforces E[eps]=0",
                     abs(mean_eps_check) < 1e-15, abs(mean_eps_check), 1e-15))

mu_eta, var_eta, skew_eta, kurt_eta = mixture_moments(pz, mu_eta1, sigma_eta1,
                                                     1.0 - pz, mu_eta2_eff, sigma_eta2)
mu_eps, var_eps, skew_eps, kurt_eps = mixture_moments(pe, mu_eps1, sigma_eps1,
                                                     1.0 - pe, mu_eps2_eff, sigma_eps2)

print(f"  eta mixture:  mean={mu_eta:+.3e}  var={var_eta:.6f}  std={np.sqrt(var_eta):.4f}")
print(f"               skew={skew_eta:+.4f}   excess_kurt={kurt_eta:+.4f}")
print(f"  eps mixture:  mean={mu_eps:+.3e}  var={var_eps:.6f}  std={np.sqrt(var_eps):.4f}")
print(f"               skew={skew_eps:+.4f}   excess_kurt={kurt_eps:+.4f}")

# Reference values from LABOUR.md
results.append(check("eta excess kurt approx +1.42",
                     abs(kurt_eta - 1.42) < 5e-3, abs(kurt_eta - 1.42), 5e-3))
results.append(check("eps excess kurt approx +52.2",
                     abs(kurt_eps - 52.2) < 5e-2, abs(kurt_eps - 52.2), 5e-2))

# Variance of stationary z = Var(eta) / (1 - rho^2)
var_z_target = var_eta / (1.0 - MODEL.rho ** 2)
print(f"  Stationary Var(z) = Var(eta)/(1-rho^2) = {var_z_target:.4f}, std = {np.sqrt(var_z_target):.4f}")

section_summary(results, "B.1 mixture parameters")


### Â§B.2  Eta quadrature moments  (Tier 1)

Sweep `n_eta in {2, 3, 4, 5, 6}` against closed-form mixture moments.

The quadrature is now a Judd (1998) construction directly on the *mixture*
density: `n_eta` is the **total** node count (no longer per-component K),
and the polynomial-exactness order against the mixture is `2 * n_eta - 1`.
At `n_eta = 3` the rule is exact through the 5th moment; at `n_eta = 5`
through the 9th.


In [ ]:
# B.2  Eta quadrature moments  (Judd-mixture, total node count)
results = []
mu_eta_truth, var_eta_truth, skew_eta_truth, kurt_eta_truth = mixture_moments(
    pz, mu_eta1, sigma_eta1, 1.0 - pz, mu_eta2_eff, sigma_eta2)

eta_rows = []
for n_eta in (2, 3, 4, 5, 6):
    e_n, e_w = get_eta_quadrature_mixture(MODEL, n_nodes=n_eta)
    sw = float(e_w.sum())
    m_, v_, s_, k_ = quad_moments(e_n, e_w)
    eta_rows.append({
        "n_eta": n_eta,
        "poly_exact_to": 2 * n_eta - 1,
        "sum w": sw,
        "min w": float(e_w.min()),
        "mean": m_,
        "var_err": abs(v_ - var_eta_truth),
        "skew": s_,
        "kurt_excess": k_,
    })
    results.append(check(f"n_eta={n_eta} sum w == 1", abs(sw - 1.0) < 1e-13, abs(sw - 1.0), 1e-13))
    results.append(check(f"n_eta={n_eta} weights positive", float(e_w.min()) > 0,
                         float(e_w.min()), 0.0))
    results.append(check(f"n_eta={n_eta} mean == 0", abs(m_) < 1e-13, abs(m_), 1e-13))
    if n_eta >= 2:
        results.append(check(f"n_eta={n_eta} variance match",
                             abs(v_ - var_eta_truth) < 1e-12,
                             abs(v_ - var_eta_truth), 1e-12))
    if n_eta >= 3:
        # Skew = m3 - 3*m1*var - m1^3, all standardized; n=3 => exact through m_5,
        # so standardized skew is exact to ~1e-10
        results.append(check(f"n_eta={n_eta} skew match",
                             abs(s_ - skew_eta_truth) < 1e-10,
                             abs(s_ - skew_eta_truth), 1e-10))
        results.append(check(f"n_eta={n_eta} excess kurt match",
                             abs(k_ - kurt_eta_truth) < 1e-10,
                             abs(k_ - kurt_eta_truth), 1e-10))

eta_df = pd.DataFrame(eta_rows)
print(eta_df.to_string(index=False, float_format=lambda v: f"{v:.5e}"))
print(f"\nTruth (analytic mixture):  var={var_eta_truth:.6f}  "
      f"skew={skew_eta_truth:+.4f}  excess_kurt={kurt_eta_truth:+.4f}")

# Polynomial exactness sweep against the mixture itself
print("\nMixture polynomial-exactness check:")
print("  Judd n-point rule integrates monomials of degree <= 2n-1 EXACTLY against")
print("  the mixture density.  Truth m_k computed from the closed-form mixture.")
from math import comb
def _mix_raw_moment(k):
    out = 0.0
    for prob, mu, s in zip([pz, 1.0 - pz], [mu_eta1, mu_eta2_eff],
                            [sigma_eta1, sigma_eta2]):
        # E[X^k] for N(mu, s^2)
        for j in range(0, k + 1, 2):
            ez = 1.0
            for m_ in range(j - 1, 0, -2):
                ez *= m_
            out += prob * comb(k, j) * (mu ** (k - j)) * (s ** j) * ez
    return out

for n_eta in (3, 5):
    e_n, e_w = get_eta_quadrature_mixture(MODEL, n_nodes=n_eta)
    print(f"  n_eta={n_eta}:")
    for k in range(2 * n_eta + 2):
        truth_k = _mix_raw_moment(k)
        disc_k = float(np.sum(e_w * e_n ** k))
        if abs(truth_k) > 1e-12:
            err = abs(disc_k - truth_k) / abs(truth_k)
            label = "rel_err"
        else:
            err = abs(disc_k - truth_k)
            label = "abs_err"
        marker = "  EXACT" if err < 1e-10 else "  approx"
        print(f"    k={k:2d}  {label} = {err:.2e}{marker}")

section_summary(results, "B.2 eta quadrature moments")


### Â§B.3  Eps quadrature moments  (Tier 1)

Same structure as Â§B.2 with the calibrated `eps` mixture. Excess kurtosis is
~+52 (the binding accuracy constraint at high Î³ â€” see Â§C.5).


In [ ]:
# B.3  Eps quadrature moments  (Judd-mixture, total node count)
results = []
mu_eps_truth, var_eps_truth, skew_eps_truth, kurt_eps_truth = mixture_moments(
    pe, mu_eps1, sigma_eps1, 1.0 - pe, mu_eps2_eff, sigma_eps2)

eps_rows = []
for n_eps in (2, 3, 4, 5, 6):
    e_n, e_w = get_eps_quadrature_corrected(MODEL, n_nodes=n_eps)
    sw = float(e_w.sum())
    m_, v_, s_, k_ = quad_moments(e_n, e_w)
    eps_rows.append({
        "n_eps": n_eps,
        "poly_exact_to": 2 * n_eps - 1,
        "sum w": sw,
        "min w": float(e_w.min()),
        "mean": m_,
        "var_err": abs(v_ - var_eps_truth),
        "skew": s_,
        "kurt_excess": k_,
    })
    results.append(check(f"n_eps={n_eps} sum w == 1", abs(sw - 1.0) < 1e-13, abs(sw - 1.0), 1e-13))
    results.append(check(f"n_eps={n_eps} weights positive", float(e_w.min()) > 0,
                         float(e_w.min()), 0.0))
    results.append(check(f"n_eps={n_eps} mean == 0", abs(m_) < 1e-12, abs(m_), 1e-12))
    if n_eps >= 2:
        results.append(check(f"n_eps={n_eps} variance match",
                             abs(v_ - var_eps_truth) < 1e-12,
                             abs(v_ - var_eps_truth), 1e-12))
    if n_eps >= 3:
        results.append(check(f"n_eps={n_eps} skew match",
                             abs(s_ - skew_eps_truth) < 1e-9,
                             abs(s_ - skew_eps_truth), 1e-9))
        rel_k = abs(k_ - kurt_eps_truth) / abs(kurt_eps_truth)
        results.append(check(f"n_eps={n_eps} excess kurt rel-match",
                             rel_k < 1e-10, rel_k, 1e-10))

eps_df = pd.DataFrame(eps_rows)
print(eps_df.to_string(index=False, float_format=lambda v: f"{v:.5e}"))
print(f"\nTruth (analytic mixture):  var={var_eps_truth:.6f}  "
      f"skew={skew_eps_truth:+.4f}  excess_kurt={kurt_eps_truth:+.4f}")

section_summary(results, "B.3 eps quadrature moments")


### Â§B.4  z-grid coverage and stationarity  (Tier 2)

Long Monte Carlo: draw `eta` from the *continuous* mixture (not from quadrature),
advance `z[t+1] = rho z[t] + eta[t]`, and check empirical stationary moments.


In [ ]:
# B.4  z-grid stationarity via continuous-mixture MC
# rho=0.991 -> high autocorrelation; effective sample size T*(1-rho^2)/(1+rho^2).
# Use 5M draws so the variance-estimator standard error is ~1%.
T = 5_000_000
burn = 50_000
rng_b = np.random.default_rng(20260427)

u = rng_b.uniform(0.0, 1.0, size=T)
comp1 = u < pz
eta_path = np.where(
    comp1,
    rng_b.normal(mu_eta1, sigma_eta1, size=T),
    rng_b.normal(mu_eta2_eff, sigma_eta2, size=T),
)
# AR(1) via lfilter:  z[t+1] = rho z[t] + eta[t]
z_path = lfilter([1.0], [1.0, -MODEL.rho], eta_path)
z_path = z_path[burn:]

emp_mean = float(np.mean(z_path))
emp_var  = float(np.var(z_path))
emp_std  = float(np.std(z_path))
emp_skew = float(np.mean(((z_path - emp_mean) / emp_std) ** 3))
emp_kurt_excess = float(np.mean(((z_path - emp_mean) / emp_std) ** 4) - 3.0)

var_z_target = var_eta_truth / (1.0 - MODEL.rho ** 2)
std_z_target = np.sqrt(var_z_target)

rel_var = abs(emp_var - var_z_target) / var_z_target
inside_3sigma = float(np.mean(np.abs(z_path) <= 3.0 * std_z_target))

print(f"Stationary z (theoretical):  mean=0  std={std_z_target:.4f}  var={var_z_target:.4f}")
print(f"Empirical (T={len(z_path):,}):")
print(f"  mean       = {emp_mean:+.5f}")
print(f"  std        = {emp_std:.4f}      (rel err vs target = {rel_var:.4%})")
print(f"  skew       = {emp_skew:+.4f}")
print(f"  ex.kurt    = {emp_kurt_excess:+.4f}")
print(f"  P(|z| <= 3 sigma_z) = {100*inside_3sigma:.2f}%")

# Coverage at production grid (n_z=11, n_stds=3.0)
zg = PC.z_grid
fraction_outside_grid = float(np.mean((z_path < zg[0]) | (z_path > zg[-1])))
print(f"\n  z_grid range: [{zg[0]:+.3f}, {zg[-1]:+.3f}]   n_z = {len(zg)}")
print(f"  Fraction of z-path outside z_grid: {100 * fraction_outside_grid:.4f}%")

# Histogram plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(z_path, bins=120, density=True, alpha=0.6, label="empirical z")
xs = np.linspace(z_path.min(), z_path.max(), 400)
ax.plot(xs, norm.pdf(xs, 0.0, std_z_target), "r-", label=f"N(0, {std_z_target:.3f}^2)")
for zv in zg:
    ax.axvline(zv, color="grey", alpha=0.4, lw=0.5)
ax.set_xlabel("z"); ax.set_ylabel("density"); ax.legend()
ax.set_title("Stationary z distribution vs production z-grid")
plt.tight_layout(); plt.show()

results = []
# Tolerance set against realistic AR(1) variance-estimator standard error
# (~1% at T=5M with rho=0.991).  See LABOUR.md for the analytic budget.
results.append(check("Empirical Var(z) matches Var(eta)/(1-rho^2) within 1.5%",
                     rel_var < 1.5e-2, rel_var, 1.5e-2))
section_summary(results, "B.4 z-grid stationarity")


### Â§B.5  Boundary clipping rate at production grid  (Tier 2)

For each `(iz, k_eta)`, compute `z_next = rho z[iz] + eta[k]` and report
the fraction outside `[z_grid[0], z_grid[-1]]`. Should concentrate at the
boundary rows only.


In [ ]:
# B.5  Boundary-clipping rate
n_z = len(PC.z_grid)
n_eta = len(PC.eta_nodes)
clip_mask = np.zeros((n_z, n_eta), dtype=int)
for iz in range(n_z):
    for k in range(n_eta):
        z_next = MODEL.rho * PC.z_grid[iz] + PC.eta_nodes[k]
        if z_next < PC.z_grid[0] or z_next > PC.z_grid[-1]:
            clip_mask[iz, k] = 1

per_iz = (clip_mask * PC.eta_weights[None, :]).sum(axis=1)
print("Clipping fraction by z-grid index (weighted by eta_weights):")
for iz in range(n_z):
    print(f"  iz={iz:2d}  z={PC.z_grid[iz]:+.3f}  clipped frac = {per_iz[iz]:.4f}"
          + ("   <-- boundary row" if iz in (0, n_z - 1) else ""))

fig, ax = plt.subplots(figsize=(7, 3.5))
im = ax.imshow(clip_mask, aspect="auto", cmap="Greys", origin="lower")
ax.set_xlabel("eta-quadrature node index"); ax.set_ylabel("z-grid index iz")
ax.set_title("Boundary clipping (1 = z' falls outside z_grid)  | n_z=%d  n_eta=%d" % (n_z, n_eta))
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

# Verdict: clipping should be concentrated at iz=0 and iz=n_z-1 only
results = []
interior_clip = float(per_iz[1:-1].max())
results.append(check("No interior-row boundary clipping (production grid)",
                     interior_clip < 1e-12, interior_clip, 1e-12))
section_summary(results, "B.5 boundary clipping")


### Â§B.6  Income table broadcasting sanity  (Tier 1)

`working_income.shape == (n_age, n_z, n_eps)`. Eight hand-coded probes
spanning `(young/peak/late) x (low/mid/high z) x (rare/common eps)`.
Pension constant in age, with AIME cap binding at the top of `z_grid`.


In [ ]:
# B.6  Income table broadcasting
from model import disposable_income_working, compute_pension_after_tax

results = []
wi = PC.working_income
pa = PC.pension_after_tax
results.append(check(f"working_income shape == (n_age, n_z, n_eps) = "
                     f"({len(PC.ages)}, {len(PC.z_grid)}, {len(PC.eps_nodes)})",
                     wi.shape == (len(PC.ages), len(PC.z_grid), len(PC.eps_nodes))))

# 8 probes covering young/peak/late * low/mid/high z * rare/common eps
ages = PC.ages
zg = PC.z_grid
eg = PC.eps_nodes
n_z = len(zg); n_eps = len(eg)

# Identify rare vs common eps (small-variance component vs large-variance)
# In get_eps_quadrature_corrected, first K_eps nodes are component 1 (rare large shocks).
# eps_nodes[0:K] = component 1 (mu_eps1, sigma_eps1)  [rare, large variance]
# eps_nodes[K:]  = component 2 (mu_eps2_eff, sigma_eps2)  [common, small variance]
K_eps = PROD_DISC.n_eps_nodes
common_idx = K_eps + (n_eps - K_eps) // 2          # mid-common
rare_idx   = 0                                      # extreme of component 1

probes = [
    ("young+lowz+common",  np.argmin(np.abs(ages - 25)), 0,            common_idx),
    ("young+highz+rare",   np.argmin(np.abs(ages - 25)), n_z - 1,      rare_idx),
    ("peak+midz+common",   np.argmin(np.abs(ages - 45)), n_z // 2,     common_idx),
    ("peak+highz+common",  np.argmin(np.abs(ages - 45)), n_z - 1,      common_idx),
    ("peak+lowz+rare",     np.argmin(np.abs(ages - 45)), 0,            rare_idx),
    ("late+midz+common",   np.argmin(np.abs(ages - 60)), n_z // 2,     common_idx),
    ("late+highz+rare",    np.argmin(np.abs(ages - 60)), n_z - 1,      rare_idx),
    ("late+lowz+common",   np.argmin(np.abs(ages - 60)), 0,            common_idx),
]

print(f"{'probe':24}  {'age':>4}  {'z':>8}  {'eps':>8}  {'gross':>10}  {'table':>10}  {'recompute':>10}")
all_match = True
for name, t, iz, ie in probes:
    age = int(ages[t]); z = float(zg[iz]); e = float(eg[ie])
    f_age = MODEL.b0 + MODEL.b1 * age + MODEL.b2 * age ** 2 / 10.0 + MODEL.b3 * age ** 3 / 100.0
    y_gross = float(np.exp(f_age + z + e))
    y_net_tab = float(wi[t, iz, ie])
    y_net_rec = float(disposable_income_working(np.array([y_gross]))[0])
    bit_match = abs(y_net_tab - y_net_rec) < 1e-15 * max(1.0, abs(y_net_rec))
    all_match = all_match and bit_match
    flag = "" if bit_match else " <-- MISMATCH"
    print(f"{name:24}  {age:4d}  {z:+8.3f}  {e:+8.3f}  {y_gross:10.4f}  {y_net_tab:10.4f}  {y_net_rec:10.4f}{flag}")
results.append(check("8 probes bit-exact vs disposable_income_working", all_match))

# Pension constant in age
const_in_age = bool(np.allclose(pa - pa[0:1, :], 0.0, atol=0.0))
results.append(check("pension_after_tax[t, iz] constant in t", const_in_age))

# AIME cap at top z grid points
aime = np.minimum(np.exp(zg) * PC.avg_det, 2.5)
top_capped = bool(aime[-1] >= 2.5 - 1e-12)
results.append(check("AIME caps at top z grid point", top_capped, float(aime[-1])))
print(f"  AIME values across z_grid: {aime.round(4)}")

section_summary(results, "B.6 income table broadcasting")


## Â§C  Convergence studies

The "what do I gain/lose" question per dial. Cheap integrand-only sweeps
always run; full-solver sweeps are gated by `RUN_FULL_SOLVER_SWEEPS`
defined in Â§0 (default `False` to keep the notebook bounded).

Result accumulator `SUMMARY` collects rows for the Â§C.8 Pareto plot.


In [ ]:
# Â§C  Result accumulator + reference utilities
SUMMARY = []  # rows: dict(dial=..., setting=..., wall_s=..., metric=..., notes=...)


def time_solver(disc_cfg, model=MODEL, label=""):
    """Run the solver and return (C, S, B, diag, wall_s)."""
    pc_local = Precompute(model, disc_cfg, verbose=False)
    t0 = time.time()
    C, S, B, diag = run_lifecycle_solver(model, pc_local, solver_config=SolverConfig(), verbose=0)
    return pc_local, C, S, B, diag, time.time() - t0


def _interpolate_wealth(pol, w_grid, w_target):
    """Linear-interp policy in wealth dim."""
    return float(np.interp(w_target, w_grid, pol))


def policy_at_reference(pc_local, C, S, B, refs):
    """refs: list of (age, z, s_vec, w) tuples.  Returns (n_refs, 3) c/W, alpha_s, alpha_b."""
    out = []
    for age, z_t, s_vec, w in refs:
        t = age - MODEL.start_age
        # nearest z-grid index
        iz = int(np.argmin(np.abs(pc_local.z_grid - z_t)))
        # nearest financial-state index (Euclidean in s-space)
        i_s = int(np.argmin(np.linalg.norm(pc_local.state_grid - s_vec, axis=1)))
        c = _interpolate_wealth(C[t, iz, i_s], pc_local.wealth_grid, w)
        a_s = _interpolate_wealth(S[t, iz, i_s], pc_local.wealth_grid, w)
        a_b = _interpolate_wealth(B[t, iz, i_s], pc_local.wealth_grid, w)
        out.append((c / w, a_s, a_b))
    return np.array(out)


# Reference states for Â§C policy comparisons (age, z, s_vec, wealth)
W_MED = 5.0
REF_STATES = [
    (40, 0.0, PC.state_grid_mu_s.copy(), W_MED),
    (65, 0.0, PC.state_grid_mu_s.copy(), W_MED),
    (40, 0.0, PC.state_grid_mu_s + np.array([2.0 * PC.state_grid_sigma_z[0], 0.0, 0.0]), W_MED),
]
print("Reference states for policy comparison:")
for r in REF_STATES:
    print(f"  age={r[0]}  z={r[1]:+.2f}  s={r[2].round(4)}  W={r[3]:.2f}")


### Â§C.1  State grid resolution N  (full solver, gated)

Sweep `N in {5, 7, 9}` per `mode in {principal, lyapunov-axis, naive}`.
Reference (gold) row is N=9 in principal mode.


In [ ]:
# C.1  State grid N sweep  (gated)
if RUN_FULL_SOLVER_SWEEPS:
    N_LIST = [5, 7, 9]                # truncated for runtime; extend to 11 if budget allows
    MODES = ["principal", "lyapunov-axis", "naive"]
    rows = []
    pol_records = {}
    for mode in MODES:
        for N in N_LIST:
            disc = PROD_DISC._replace(state_grid_sizes=(N, N, N), state_grid_mode=mode)
            pc_l, C, S, B, diag, wall = time_solver(disc, label=f"{mode}/N={N}")
            pol = policy_at_reference(pc_l, C, S, B, REF_STATES)
            pol_records[(mode, N)] = pol
            rows.append({"mode": mode, "N": N, "wall_s": wall,
                         "newton_failrate": diag["total_newton_failures"] / max(diag["total_calls"], 1)})
            print(f"  [{mode}, N={N}]  wall = {wall:.1f}s   policy(c/W,as,ab) at age=40 mid: {pol[0].round(3)}")

    # Reference: principal at largest N
    ref_pol = pol_records[("principal", N_LIST[-1])]
    for mode in MODES:
        for N in N_LIST:
            err = float(np.max(np.abs(pol_records[(mode, N)] - ref_pol)))
            for r in rows:
                if r["mode"] == mode and r["N"] == N:
                    r["max_pol_err_vs_gold"] = err
                    SUMMARY.append({"dial": "state_grid_N", "setting": f"{mode}/N={N}",
                                    "wall_s": r["wall_s"], "metric": err})

    df = pd.DataFrame(rows)
    print(); print(df.to_string(index=False, float_format=lambda v: f"{v:.4e}"))

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for mode in MODES:
        sub = df[df["mode"] == mode].sort_values("N")
        ax.semilogy(sub["N"], sub["max_pol_err_vs_gold"].clip(lower=1e-15), "o-", label=mode)
    ax.set_xlabel("N (per-axis)"); ax.set_ylabel("max policy err vs principal/N=9")
    ax.set_title("Â§C.1  State-grid convergence"); ax.legend()
    plt.tight_layout(); plt.show()
else:
    print("Skipped (RUN_FULL_SOLVER_SWEEPS=False).  Enable in Â§0 to run the full sweep.")


### Â§C.2  State quadrature K_s  (integrand-only)

Integrand `f(v) = exp(a Â· v)` with closed-form expectation
`E[f(v)] = exp(0.5 a^T Sigma_ss a)` under `v ~ N(0, Sigma_ss)`.
This isolates quadrature accuracy from solver accuracy.


In [ ]:
# C.2  K_s integrand error
a_vec = np.array([0.5, 0.5, 0.5])
analytic_v = float(np.exp(0.5 * a_vec @ Sigma_ss @ a_vec))
rows = []
for K_s in (2, 3, 4, 5):
    v_n, v_w = get_state_quadrature(MODEL, n_nodes=K_s)
    gh = float(np.sum(v_w * np.exp(v_n @ a_vec)))
    rel = abs(gh - analytic_v) / analytic_v
    rows.append({"K_s": K_s, "K_total": K_s ** 3, "gh": gh, "rel_err": rel})
    SUMMARY.append({"dial": "K_s", "setting": K_s, "wall_s": np.nan, "metric": rel})

df = pd.DataFrame(rows)
print(df.to_string(index=False, float_format=lambda v: f"{v:.6e}"))

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(df["K_total"], df["rel_err"].clip(lower=1e-17), "o-")
ax.set_xlabel("K_s^3 = total state quadrature nodes"); ax.set_ylabel("rel err on E[exp(a.v)]")
ax.set_title("Â§C.2  State quadrature integrand error"); plt.tight_layout(); plt.show()


### Â§C.3  Return residual quadrature K_r  (integrand-only)

Same form, against `Sigma_r_cond`.


In [ ]:
# C.3  K_r integrand error
a_vec = np.array([0.5, 0.5, 0.5])
analytic_r = float(np.exp(0.5 * a_vec @ Sigma_r_cond @ a_vec))
rows = []
for K_r in (2, 3, 4, 5):
    r_n, r_w = get_return_quadrature(MODEL, n_nodes=K_r)
    gh = float(np.sum(r_w * np.exp(r_n @ a_vec)))
    rel = abs(gh - analytic_r) / analytic_r
    rows.append({"K_r": K_r, "K_total": K_r ** 3, "gh": gh, "rel_err": rel})
    SUMMARY.append({"dial": "K_r", "setting": K_r, "wall_s": np.nan, "metric": rel})

df = pd.DataFrame(rows)
print(df.to_string(index=False, float_format=lambda v: f"{v:.6e}"))

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(df["K_total"], df["rel_err"].clip(lower=1e-17), "o-")
ax.set_xlabel("K_r^3 = total return-residual nodes"); ax.set_ylabel("rel err on E[exp(a.r)]")
ax.set_title("Â§C.3  Return-residual quadrature integrand error"); plt.tight_layout(); plt.show()


### Â§C.4  Eta quadrature `n_eta`  (CRRA integrand)

Stress test the rule on the actual FOC integrand: with `c = exp(z + Î·)` and
`z = 0`, `E[c^{1-Î³}] = E[exp(Î»Â·Î·)]` for `Î» = 1 - Î³`. Truth is the closed-form
normal-mixture MGF.

This is the most economist-relevant view: it shows whether the rule, beyond
matching polynomial moments, also captures the exponential tail mass that
drives precautionary savings at high Î³.


In [ ]:
# C.4  n_eta sweep on the CRRA integrand  E[exp((1-gamma) eta)]
rows = []
for gamma in (3.0, 4.0, 6.0, 8.0, 10.0):
    lam = 1.0 - gamma
    truth = (pz * np.exp(lam * mu_eta1 + 0.5 * lam ** 2 * sigma_eta1 ** 2)
             + (1 - pz) * np.exp(lam * mu_eta2_eff + 0.5 * lam ** 2 * sigma_eta2 ** 2))
    for n_eta in (2, 3, 4, 5, 6, 8):
        e_n, e_w = get_eta_quadrature_mixture(MODEL, n_nodes=n_eta)
        approx = float(np.sum(e_w * np.exp(lam * e_n)))
        rel = abs(approx - truth) / truth
        rows.append({"gamma": gamma, "n_eta": n_eta,
                     "truth": truth, "approx": approx, "rel_err": rel})
        if gamma == 6.0:
            SUMMARY.append({"dial": "n_eta", "setting": n_eta,
                            "wall_s": np.nan, "metric": rel})

df_eta_crra = pd.DataFrame(rows)
print(df_eta_crra.to_string(index=False, float_format=lambda v: f"{v:.4e}"))

fig, ax = plt.subplots(figsize=(7, 4.5))
for g_ in sorted(df_eta_crra["gamma"].unique()):
    sub = df_eta_crra[df_eta_crra["gamma"] == g_]
    ax.semilogy(sub["n_eta"], sub["rel_err"].clip(lower=1e-17),
                "o-", label=f"Î³={g_}")
ax.set_xlabel("n_eta (total Judd nodes)")
ax.set_ylabel("rel err on E[exp((1-Î³) Î·)]")
ax.set_title("Â§C.4  Eta Judd quadrature: CRRA integrand error")
ax.legend()
plt.tight_layout(); plt.show()

# Diagnostic: production default n_eta=3 at Î³=6 â€” note vs older K=3 (6 GH nodes).
prod_row = df_eta_crra[(df_eta_crra["gamma"] == 6.0) & (df_eta_crra["n_eta"] == 3)].iloc[0]
print(f"\nProduction n_eta=3 at Î³=6:  rel err = {prod_row['rel_err']:.3e}")
print("(For Î³ â‰¥ 5 with high-precision FOC, set n_eta_nodes=5 â€” see audit_judd_economist.py.)")


### Â§C.5  Eps quadrature `n_eps`  (CRRA integrand)

Same stress test for `eps`. Excess kurtosis +52 makes this the binding
constraint at high Î³: even Judd `n_eps = 8` does not eliminate the error
at Î³ = 8. The economist takeaway is to keep Î³ â‰¤ 5 if running at the
default `n_eps = 3`.


In [ ]:
# C.5  n_eps sweep on the CRRA integrand  E[exp((1-gamma) eps)]
rows = []
for gamma in (3.0, 4.0, 6.0, 8.0):
    lam = 1.0 - gamma
    truth = (pe * np.exp(lam * mu_eps1 + 0.5 * lam ** 2 * sigma_eps1 ** 2)
             + (1 - pe) * np.exp(lam * mu_eps2_eff + 0.5 * lam ** 2 * sigma_eps2 ** 2))
    for n_eps in (2, 3, 4, 5, 6, 8):
        e_n, e_w = get_eps_quadrature_corrected(MODEL, n_nodes=n_eps)
        approx = float(np.sum(e_w * np.exp(lam * e_n)))
        rel = abs(approx - truth) / truth
        rows.append({"gamma": gamma, "n_eps": n_eps,
                     "truth": truth, "approx": approx, "rel_err": rel})
        if gamma == 6.0:
            SUMMARY.append({"dial": "n_eps", "setting": n_eps,
                            "wall_s": np.nan, "metric": rel})

df_eps_crra = pd.DataFrame(rows)
print(df_eps_crra.to_string(index=False, float_format=lambda v: f"{v:.4e}"))

fig, ax = plt.subplots(figsize=(7, 4.5))
for g_ in sorted(df_eps_crra["gamma"].unique()):
    sub = df_eps_crra[df_eps_crra["gamma"] == g_]
    ax.semilogy(sub["n_eps"], sub["rel_err"].clip(lower=1e-17),
                "o-", label=f"Î³={g_}")
ax.set_xlabel("n_eps (total Judd nodes)")
ax.set_ylabel("rel err on E[exp((1-Î³) Îµ)]")
ax.set_title("Â§C.5  Eps Judd quadrature: CRRA integrand error  (heavy tail)")
ax.legend()
plt.tight_layout(); plt.show()

# Reality check at Î³ â‰¥ 5: even n_eps=6 leaves significant residual error.
print("\nRel err at Î³=8 across n_eps:")
for r in df_eps_crra[df_eps_crra["gamma"] == 8.0].itertuples():
    print(f"  n_eps={r.n_eps:2d}:  rel_err = {r.rel_err:.3e}")
print("=> No low-n rule resolves the eps tail at Î³=8. Reduce Î³ or accept the bias.")


### Â§C.6  Income grid n_z  (full solver, gated)

`n_z in {7, 11}` (truncated from {7, 9, 11, 13, 15} for runtime).  The
solver dominantly interpolates consumption linearly in z, so this is the
expected binding accuracy constraint.


In [ ]:
# C.6  n_z sweep
if RUN_FULL_SOLVER_SWEEPS:
    NZ_LIST = [7, 11]      # extend to [7, 9, 11, 13, 15] if budget allows
    rows = []
    pol_records = {}
    for nz in NZ_LIST:
        disc = PROD_DISC._replace(n_z=nz)
        pc_l, C, S, B, diag, wall = time_solver(disc, label=f"n_z={nz}")
        pol_records[nz] = policy_at_reference(pc_l, C, S, B, REF_STATES)
        rows.append({"n_z": nz, "wall_s": wall})
        print(f"  [n_z={nz}]  wall = {wall:.1f}s")
    ref_pol = pol_records[NZ_LIST[-1]]
    for r in rows:
        err = float(np.max(np.abs(pol_records[r["n_z"]] - ref_pol)))
        r["max_pol_err"] = err
        SUMMARY.append({"dial": "n_z", "setting": r["n_z"], "wall_s": r["wall_s"], "metric": err})
    df = pd.DataFrame(rows)
    print(df.to_string(index=False, float_format=lambda v: f"{v:.4e}"))
else:
    print("Skipped (RUN_FULL_SOLVER_SWEEPS=False).")


### Â§C.7  Wealth grid n_w  (full solver, gated)

`n_w in {75, 150}` (truncated from {75, 150, 300}).


In [ ]:
# C.7  n_w sweep
if RUN_FULL_SOLVER_SWEEPS:
    NW_LIST = [75, 150]    # extend to [75, 150, 300] if budget allows
    rows = []
    pol_records = {}
    for nw in NW_LIST:
        disc = PROD_DISC._replace(n_wealth=nw, n_savings=nw)
        pc_l, C, S, B, diag, wall = time_solver(disc, label=f"n_w={nw}")
        pol_records[nw] = policy_at_reference(pc_l, C, S, B, REF_STATES)
        rows.append({"n_w": nw, "wall_s": wall})
        print(f"  [n_w={nw}]  wall = {wall:.1f}s")
    ref_pol = pol_records[NW_LIST[-1]]
    for r in rows:
        err = float(np.max(np.abs(pol_records[r["n_w"]] - ref_pol)))
        r["max_pol_err"] = err
        SUMMARY.append({"dial": "n_w", "setting": r["n_w"], "wall_s": r["wall_s"], "metric": err})
    df = pd.DataFrame(rows)
    print(df.to_string(index=False, float_format=lambda v: f"{v:.4e}"))
else:
    print("Skipped (RUN_FULL_SOLVER_SWEEPS=False).")


### Â§C.8  Pareto cost-accuracy summary

Every dial setting collected above plotted on (cost, error) axes.


In [ ]:
# C.8  Pareto summary
df_sum = pd.DataFrame(SUMMARY)
print("Collected sweep results:")
print(df_sum.to_string(index=False, float_format=lambda v: f"{v:.4e}"))

# Plot integrand-only sweeps (C.2-C.5) on a single accuracy-axis chart
fig, ax = plt.subplots(figsize=(8, 5))
for dial in df_sum["dial"].unique():
    sub = df_sum[df_sum["dial"] == dial].sort_values("setting")
    ax.semilogy(sub["setting"], sub["metric"].clip(lower=1e-17), "o-", label=dial)
ax.set_xlabel("dial setting"); ax.set_ylabel("error metric (rel for integrand, max for full solver)")
ax.set_title("Â§C.8  Cost-accuracy summary across dials"); ax.legend()
plt.tight_layout(); plt.show()

if RUN_FULL_SOLVER_SWEEPS and df_sum["wall_s"].notna().any():
    sub = df_sum[df_sum["wall_s"].notna()]
    fig, ax = plt.subplots(figsize=(8, 5))
    for dial in sub["dial"].unique():
        s = sub[sub["dial"] == dial]
        ax.loglog(s["wall_s"], s["metric"].clip(lower=1e-17), "o-", label=dial)
    ax.set_xlabel("solver wall time (s)"); ax.set_ylabel("policy err vs gold")
    ax.set_title("Pareto frontier (full-solver sweeps only)"); ax.legend()
    plt.tight_layout(); plt.show()


## Â§D  Conclusions

### Per-dial recommendations

| Dial | Production | Reason |
|------|------------|--------|
| `state_grid_sizes` | `(7,7,7)` principal | Coverage: principal mode at `n_stds=3.0` covers ~99% of historical sample at 0.43x the naive volume. Smooth-function L_inf error decays cleanly as O(1/NÂ²). |
| `n_state_quad_nodes` | `K_s = 3` | Integrand error on `exp(aÂ·v)` at `K_s=3` typically `< 1e-8`; pushing to `K_s=5` is dominated by `K_r` and grid resolution. |
| `n_ret_nodes_1d`    | `K_r = 3` | Similar story for `Sigma_r_cond`. |
| `n_z`               | `n_z = 11` | Likely binding constraint per LABOUR.md; B.5 and the Â§C.6 gated sweep characterise the trade-off. |
| `n_eta_nodes`       | `n_eta = 3` (Î³ â‰¤ 4),  `n_eta = 5` (Î³ â‰¥ 5) | Judd-mixture rule, `n_eta` is the **total** node count. Polynomial exactness `2n_eta - 1`. At Î³ â‰¤ 4 the n=3 rule has rel err `< 1e-3` on the CRRA integrand. At Î³ â‰¥ 5 opt up to n=5 (rel err `< 1e-5` at Î³=10). |
| `n_eps_nodes`       | `n_eps = 3` (Î³ â‰¤ 3),  `n_eps â‰¥ 5` (Î³ â‰¥ 4) | Excess kurtosis +52 is the binding constraint. **Even n=8 leaves > 50% error at Î³=8** â€” neither Judd nor the previous stratified-GH method resolves the eps tail at high Î³. Reduce Î³ or accept the bias. |
| `n_wealth`          | `n_w = 150` | EGM-driven; geometric grid concentrates near zero where MU is steep. |

### Surprises and open issues

- **Migration from concatenated GH-mixture to Judd-mixture**: same polynomial-
  exactness order at half (marginal) / a quarter (joint) the node count. For
  `n_eta = n_eps = 3`, total joint nodes drop from 36 to 9 inside the working-
  age FOC.
- **Per-dial accuracy at high Î³**: at the same poly-exactness order (5), the
  *old* per-component K=3 rule was ~10Ã— more accurate on the Î· CRRA integrand
  than Judd `n=3`, because GH stratification puts more nodes in each
  component's tail. Going to Judd `n=5` (still fewer nodes than old K=3)
  recovers and exceeds old accuracy. See `tests/audit_judd_economist.py`.
- **Eps tail is unsolvable at high Î³ with low-n rules**: a finding the
  notebook now confirms in Â§C.5. This is a property of the eps mixture's
  excess kurtosis (+52), not Judd-specific. The previous expectation
  that `n_eps = 5` was sufficient at Î³ â‰¥ 5 is incorrect â€” relative errors
  are ~50% even with the new exactness-order-9 rule at n=5.
